# External image: PNG in, result out (#11)

Runs a trained model on **any image file** (PNG, JPG): typically a screenshot
of a published image suspected of tampering, such as the Vertical 52 /
Kronstadt case. **One cell does everything**: set the two paths, run it.

**Out-of-domain test, for illustration only.** The model was trained on
Sentinel-1 VV amplitude patches; a screenshot is usually optical, resampled
and compressed. The result has **no metric value**: it shows what the model
reacts to, not how well it works (the metrics are in `docs/report.md`).

What the cell does (details in `README.md` and `external_image.py`):
1. reads the image, grey = **float mean of R, G, B** (no binarization);
2. gives it the mean and standard deviation, in dB, of the model's training
   patches (`config.json`), so that the network sees the range it knows;
3. cuts it into 512x512 windows overlapping by half, predicts each, keeps
   the highest probability per pixel;
4. image decision: forged if at least one window reaches the image threshold
   frozen on val (`eval_level<T>_trained<T>/settings.json`);
5. shows input | probability map | binary map at the frozen pixel threshold,
   and saves the figure and a JSON summary to Drive.

Needs the run folder on Drive with `best.pt`, `config.json` and the settings
of its own-level evaluation. **Runtime: GPU** (CPU works, slower).

In [ ]:
# ---- Parameters (your values) ---------------------------------------------
REPO_URL = "https://github.com/bonlieuguillaume/aletheia.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/aletheia"

IMAGE_PATH = DRIVE_ROOT + "/external/vertical52_kronstadt.png"   # your PNG / JPG on Drive
RUN_NAME = "v0.1_level3_resnet34"                               # reference model of v0.1
RUN_DIR = DRIVE_ROOT + "/runs/" + RUN_NAME
OUTPUT_DIR = RUN_DIR + "/external"                              # figure + JSON written here

# ---- Drive, code, packages (on the Colab VM: nothing reaches your PC) ------
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
from google.colab import drive

drive.mount("/content/drive")
REPO_DIR = "/content/aletheia"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run(["pip", "install", "-q", "-r", REPO_DIR + "/requirements.txt"], check=True)
os.chdir(REPO_DIR + "/features/external_image")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

from external_image import analyse_image, plot_result
from features.evaluation.evaluation import describe_score_method
from features.unet.unet import load_model

# ---- Model, with the settings frozen on the val of its own level -----------
model, config = load_model(RUN_DIR)
print(f"Model {RUN_NAME}, trained at commit {config.get('git_commit')}; code at commit {commit}")

# ---- Prediction -------------------------------------------------------------
result = analyse_image(model, config, IMAGE_PATH)
decision = "FORGED" if result["is_forged"] else "authentic"
height, width = result["gray"].shape
print(f"Image {os.path.basename(IMAGE_PATH)}: {width} x {height} px, {len(result['windows'])} windows of 512")
print(describe_score_method(result["score_method"]) + ", per window; image score = highest window")
print(f"Score {result['score']:.3f} -> {decision} (threshold {result['image_threshold']:.3f})")
print(f"Zone called forged: {result['zone'].mean() * 100:.2f} % of the image")
print("OUT-OF-DOMAIN test (not Sentinel-1): for illustration only, no metric value.")

# ---- Figure and summary, saved to Drive ---------------------------------------
name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
title = (f"{name}: score {result['score']:.3f} -> {decision} "
         f"(threshold {result['image_threshold']:.3f}, {RUN_NAME})  -  out-of-domain test")
fig = plot_result(result, title)
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(f"{OUTPUT_DIR}/{name}.png", dpi=150, bbox_inches="tight")
summary = {
    "image": IMAGE_PATH,
    "run": RUN_NAME,
    "code_commit": commit,
    "score": result["score"],
    "decision": decision,
    "score_method": result["score_method"],
    "image_threshold": result["image_threshold"],
    "pixel_threshold": result["pixel_threshold"],
    "zone_fraction": float(result["zone"].mean()),
    "best_window": result["best_window"],
}
with open(f"{OUTPUT_DIR}/{name}.json", "w") as file:
    json.dump(summary, file, indent=2)
print(f"Saved: {OUTPUT_DIR}/{name}.png and .json")
plt.show()